https://artificialanalysis.ai/models
- interleaved reasoning/thinking
    - https://arxiv.org/abs/2505.19640
        - nana banana：Interleaved Generation
    - https://moonshotai.github.io/Kimi-K2/thinking.html
        - Kimi K2 Thinking can execute up to 200 – 300 sequential tool calls without human interference, reasoning **coherently** across hundreds of steps to solve complex problems.
    - turn or step
        - conversation > (user) turn > (agent) steps
            - reasoning -> tool call -> tool result -> reasoning
                - coherent reasoning
        - https://huggingface.co/moonshotai/Kimi-K2-Thinking/commit/603577facb43f3c1153c7471684735a6da9dc873
- Built as a thinking agent, it **reasons step by step while using tools**, achieving state-of-the-art performance on ...
    - end-to-end trained to interleave chain-of-thought reasoning with function calls
- interleaved reasoning: 交错推理（nano banana 交错生成）
    - `<cot, ans>`: 大多数模型会在一次长推理里想完所有步骤，然后给出答案；前面一步错了，后面全错。
    - `<think, ans> * n`: 它先想一小段，然后调用工具（搜索/写代码/操作终端等），看结果，再接着想，如此循环。
        - 每一 step（`<think, ans>`）都意味着可以定义和计算一次 reward
        - plan -> act -> verify -> reflect -> refine
            - 必须是有状态的（for plan & refine）；
        - By reasoning while actively using a diverse set of tools, K2 Thinking is capable of **planning, reasoning, executing, and adapting** across hundreds of steps to tackle some of the most challenging academic and analytical problems. In one instance, it successfully solved a PhD-level mathematics problem through ​23 interleaved reasoning and tool calls​, exemplifying its capacity for deep, structured reasoning and long-horizon problem solving:
    - 内生 interleave，非外部编排（ReAct）；
- QAT (Quantization-Aware Training) & MoE
    - Low-bit quantization is an effective way to reduce inference latency and GPU memory usage on large-scale inference servers. However, thinking models use excessive decoding lengths, and thus quantization often results in substantial performance drops.
To overcome this challenge, we adopt Quantization-Aware Training (QAT) during the post-training phase, applying INT4 weight-only quantization to the MoE components. It allows K2 Thinking to support native INT4 inference with a roughly 2x generation speed improvement while achieving state-of-the-art performance. All benchmark results are reported under INT4 precision.

- misc
    - https://www.minimaxi.com/news/why-is-interleaved-thinking-important-for-m2
    - https://docs.claude.com/en/docs/build-with-claude/extended-thinking#interleaved-thinking
    - https://www.kimi.com/chat/19a8caca-9602-8a73-8000-09f89db267ee

### chat template

- https://huggingface.co/moonshotai/Kimi-K2-Instruct/blob/main/chat_template.jinja
- https://huggingface.co/moonshotai/Kimi-K2-Thinking/blob/main/chat_template.jinja
- https://docs.vllm.ai/projects/recipes/en/latest/moonshotai/Kimi-K2-Think.html
    - Deep Thinking & Tool Orchestration: **End-to-end trained to interleave chain-of-thought reasoning with function calls**, enabling autonomous research, coding, and writing workflows that last hundreds of steps without drift.
- vllm interleaved thinking: https://docs.vllm.ai/en/latest/features/interleaved_thinking/
    - Interleaved thinking allows models to reason between tool calls, enabling more sophisticated decision-making after receiving tool results. This feature helps models chain multiple tool calls with reasoning steps in between and make nuanced decisions based on intermediate results.
- system, user, assistant (think, tool calls), tool results, assistant (think, tool calls), tool results
    - 每一轮新的工具调用，必须由一条新的 assistant 消息发起，

In [1]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("moonshotai/Kimi-K2-Instruct")

/home/whaow/anaconda3/envs/verl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/whaow/anaconda3/envs/verl/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


The repository moonshotai/Kimi-K2-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Instruct .
 You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


In [2]:
messages = [
    {
        "role": "user",
        "content": "请用搜索和计算帮我估算 2024 年全球光伏装机总量。"
    },
    {
        "role": "assistant",
        "content": "好的，我先查一下最近的光伏装机容量数据，然后再做一个数量级估算。",
        # 这一段只在 K2-Thinking 中真正用到
        "reasoning_content": (
            "Step 1: Use web_search to fetch latest "
            "'global solar PV installed capacity 2023/2024'.\n"
            "Step 2: Extract GW scale and growth trend.\n"
            "Step 3: Extrapolate to 2024 with a simple CAGR assumption.\n"
            "Step 4: Double-check that the final number is within typical "
            "IEA / IRENA ranges."
        ),
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "web_search",
                    "arguments": (
                        '{"query": "global solar PV installed capacity 2023 2024"}'
                    )
                }
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": '{"capacity_gw": 1600, "year": 2024}'
    },
]

tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]

In [5]:
from rich.pretty import pprint
prompt = tok.apply_chat_template(
    messages,
    tools=tools,                   # 只有支持 tools 的模板才会用到
    add_generation_prompt=True,    # 结尾加上下一条 assistant 的起始标记
    tokenize=False,                # 直接看字符串
)
print(prompt)

<|im_system|>tool_declare<|im_middle|>[{"function":{"description":"Search the web for the given query.","name":"web_search","parameters":{"properties":{"query":{"type":"string"}},"required":["query"],"type":"object"}},"type":"function"}]<|im_end|><|im_system|>system<|im_middle|>You are Kimi, an AI assistant created by Moonshot AI.<|im_end|>
<|im_user|>user<|im_middle|>请用搜索和计算帮我估算 2024 年全球光伏装机总量。<|im_end|><|im_assistant|>assistant<|im_middle|>好的，我先查一下最近的光伏装机容量数据，然后再做一个数量级估算。<|tool_calls_section_begin|><|tool_call_begin|>call_1<|tool_call_argument_begin|>{"query": "global solar PV installed capacity 2023 2024"}<|tool_call_end|><|tool_calls_section_end|><|im_end|><|im_system|>tool<|im_middle|>## Return of call_1
{"capacity_gw": 1600, "year": 2024}<|im_end|><|im_assistant|>assistant<|im_middle|>


- tool_declare: tool 声明在最前面
- tool 返回被当成 “system-like message” 写进去，并且带一个 Markdown 风格的头：`## Return of call_1`
    - 这里完全没有 <think> 或 reasoning_content 的概念 —— 所有推理都是“隐含在模型内部”，模板只负责拼历史对话 + 工具调用。

In [7]:
tok2 = AutoTokenizer.from_pretrained("moonshotai/Kimi-K2-Thinking")
prompt = tok2.apply_chat_template(
    messages,
    tools=tools,                   # 只有支持 tools 的模板才会用到
    add_generation_prompt=True,    # 结尾加上下一条 assistant 的起始标记
    tokenize=False,                # 直接看字符串
)
print(prompt)

The repository moonshotai/Kimi-K2-Thinking contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Thinking .
 You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Thinking.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


<|im_system|>tool_declare<|im_middle|>[{"function":{"description":"Search the web for the given query.","name":"web_search","parameters":{"properties":{"query":{"type":"string"}},"required":["query"],"type":"object"}},"type":"function"}]<|im_end|><|im_system|>system<|im_middle|>You are Kimi, an AI assistant created by Moonshot AI.<|im_end|><|im_user|>user<|im_middle|>请用搜索和计算帮我估算 2024 年全球光伏装机总量。<|im_end|><|im_assistant|>assistant<|im_middle|><think>Step 1: Use web_search to fetch latest 'global solar PV installed capacity 2023/2024'.
Step 2: Extract GW scale and growth trend.
Step 3: Extrapolate to 2024 with a simple CAGR assumption.
Step 4: Double-check that the final number is within typical IEA / IRENA ranges.</think>好的，我先查一下最近的光伏装机容量数据，然后再做一个数量级估算。<|tool_calls_section_begin|><|tool_call_begin|>call_1<|tool_call_argument_begin|>{"query": "global solar PV installed capacity 2023 2024"}<|tool_call_end|><|tool_calls_section_end|><|im_end|><|im_system|>tool<|im_middle|>## Return of call_

- 先找“最后一条没有 tool_calls 的 assistant 消息”（assistant 彻底完成了上次 user 的query，标记着 turn 的结束）来切分 history / suffix；
    - history 里的 assistant 都变成 `<think></think>` + content（不保留历史推理）；
        - `<|im_assistant|>assistant<|im_middle|><think></think>可见回答内容...<|im_end|>`
    - suffix 里的 assistant 会把 reasoning_content 放进 `<think>...</think>` 里。
        - `<|im_assistant|>assistant<|im_middle|><think>最近一段 reasoning_content ...</think>可见回答内容...<|im_end|>`

In [18]:
messages = [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "用户问题"},

    # 第一次：assistant 思考 + 调用工具 A
    {
        "role": "assistant",
        "content": "",
        "reasoning_content": "先调用工具 A 拿到基础信息……",
        "tool_calls": [
            {
                "id": "call_A_1",
                "type": "function",
                "function": {
                    "name": "tool_A",
                    "arguments": "{...}"
                }
            }
        ]
    },
    # 工具 A 返回
    {
        "role": "tool",
        "tool_call_id": "call_A_1",
        "name": "tool_A",
        "content": "{...A 的结果...}"
    },

    # 第二次：assistant 再思考 + 调用工具 B（串行）
    {
        "role": "assistant",
        "content": "",
        "reasoning_content": "基于 A 的结果，接下来需要调用工具 B 做补充……",
        "tool_calls": [
            {
                "id": "call_B_1",
                "type": "function",
                "function": {
                    "name": "tool_B",
                    "arguments": "{...}"
                }
            }
        ]
    },
    # 工具 B 返回
    {
        "role": "tool",
        "tool_call_id": "call_B_1",
        "name": "tool_B",
        "content": "{...B 的结果...}"
    },
]

In [19]:
tok2 = AutoTokenizer.from_pretrained("moonshotai/Kimi-K2-Thinking")
prompt = tok2.apply_chat_template(
    messages,
    tools=tools,                   # 只有支持 tools 的模板才会用到
    add_generation_prompt=True,    # 结尾加上下一条 assistant 的起始标记
    tokenize=False,                # 直接看字符串
)
print(prompt)

The repository moonshotai/Kimi-K2-Thinking contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Thinking .
 You can inspect the repository content at https://hf.co/moonshotai/Kimi-K2-Thinking.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


<|im_system|>tool_declare<|im_middle|>[{"function":{"description":"Search the web for the given query.","name":"web_search","parameters":{"properties":{"query":{"type":"string"}},"required":["query"],"type":"object"}},"type":"function"},{"function":{"description":"Run Python code for numeric estimation.","name":"python","parameters":{"properties":{"code":{"type":"string"}},"required":["code"],"type":"object"}},"type":"function"}]<|im_end|><|im_system|>system<|im_middle|>...<|im_end|><|im_user|>user<|im_middle|>用户问题<|im_end|><|im_assistant|>assistant<|im_middle|><think>先调用工具 A 拿到基础信息……</think><|tool_calls_section_begin|><|tool_call_begin|>call_A_1<|tool_call_argument_begin|>{...}<|tool_call_end|><|tool_calls_section_end|><|im_end|><|im_system|>tool_A<|im_middle|>## Return of call_A_1
{...A 的结果...}<|im_end|><|im_assistant|>assistant<|im_middle|><think>基于 A 的结果，接下来需要调用工具 B 做补充……</think><|tool_calls_section_begin|><|tool_call_begin|>call_B_1<|tool_call_argument_begin|>{...}<|tool_call_end|

### minimax m2

- last_user_index: 区分「历史轮次」和「当前 user turn」；
    - Kimi-K2-Thinking：用 last_non_tool_call_assistant 把「最新的一段推理链（可能跨用户）」标记为 suffix，只保留那里的 reasoning_content。

### 数据合成

- 构造大量 多轮 tool-use 轨迹：想一点 → call tool A → 看结果 → 想一点 → call tool B → … → 完成任务
- 把这些轨迹当成 训练样本，模型在里面学的不是“单轮回答”，而是：
    - 什么时候继续写 reasoning token？
    - 什么时候切换成 function call？
    - tool 失败时要不要反思 / 改参数 / 换工具？